# Synthetic-Content Disclosure in Generative Image Model Cards (v3, final)

Measures what the most-downloaded text-to-image models tell downstream
developers about marking their outputs.

**This version embeds the post-validation coder.** Changes from v2, all driven
by hand validation of 73 rows at precision 0.973:

- The three marking categories now require a positive, output-directed
  confirmation cue. A bare keyword match no longer counts, because "provenance"
  refers to training-corpus provenance far more often than to output provenance.
- Red-teaming and evaluation descriptions no longer count as prohibited-use
  statements.
- Gate and authentication instructions no longer count as downstream
  obligations.
- Every trial records both the naive and the refined flag, so the paper can
  report how much naive keyword auditing overstates disclosure.

**Set N_MODELS = 500 and run once.** About 20 minutes, no GPU.

## 1. Setup

In [ ]:
!pip install -q huggingface_hub pandas matplotlib

import json, re, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from huggingface_hub import HfApi, ModelCard

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
OUT = Path("mcd_out"); OUT.mkdir(exist_ok=True)
FETCH_DATE = pd.Timestamp.utcnow().strftime("%Y-%m-%d")
print("ready, fetch date", FETCH_DATE)

## 2. Fetch

In [ ]:
N_MODELS = 500          # <-- this is the line to set
TASK = "text-to-image"
api = HfApi()

def list_top(task, limit):
    for kw in (dict(task=task, sort="downloads", limit=limit),
               dict(filter=task, sort="downloads", limit=limit),
               dict(task=task, limit=limit)):
        try:
            out = list(api.list_models(**kw))
            if out:
                return out
        except TypeError:
            continue
    raise RuntimeError("no working list_models signature")

def get_card(mid):
    try:
        return ModelCard.load(mid).text
    except Exception:
        return None

models = sorted(list_top(TASK, N_MODELS), key=lambda m: m.downloads or 0, reverse=True)
print(f"listed {len(models)} models")

rows = []
for i, m in enumerate(models):
    if i % 50 == 0:
        print(f"  {i}/{len(models)}")
    rows.append({
        "model_id": m.id,
        "org": m.id.split("/")[0] if "/" in m.id else "(none)",
        "downloads": m.downloads or 0,
        "likes": m.likes or 0,
        "tags": ",".join(m.tags or []),
        "created_at": str(getattr(m, "created_at", "")),
        "card_text": get_card(m.id),
    })
    time.sleep(0.1)

raw = pd.DataFrame(rows)
raw["fetch_date"] = FETCH_DATE
raw.to_json(OUT / "raw_cards.jsonl", orient="records", lines=True)
print(f"fetched {len(raw)}; {raw['card_text'].notna().sum()} have a retrievable card")

## 3. Coding schema (post-validation)

In [ ]:
"""Coding schema v2: context-aware.

Hand validation of the v1 keyword coder found three systematic false positive
classes, all of which inflate the apparent disclosure rate:

  1. "watermark" appearing inside negative prompts, where it names an artifact
     the user wants the model to avoid drawing. This is the opposite of output
     watermarking.
  2. "provenance" referring to training-corpus provenance tracking rather than
     provenance attached to generated outputs.
  3. Synthetic-content language describing synthetic *training data*, or
     describing the "AI-generated look" as a quality defect the model reduces.

v2 keeps the v1 patterns as candidate generators, then applies an exclusion
filter over a context window around each match. Every trial records both the
raw v1 flag and the refined v2 flag, so the paper can report the gap. That gap
is itself a result: it quantifies how badly naive keyword auditing overstates
disclosure.
"""

from __future__ import annotations

import re

WINDOW = 150

CANDIDATES = {
    "watermark": [
        r"\bwatermark(?:s|ed|ing)?\b",
        r"\bsynthid\b",
        r"\bstegastamp\b",
        r"invisible[- ]watermark",
        r"\bimwatermark\b",
    ],
    "provenance": [
        r"\bc2pa\b",
        r"content\s+credential",
        r"\bprovenance\b",
        r"content\s+authenticity",
    ],
    "synthetic_disclosure": [
        r"ai[- ]generated",
        r"machine[- ]generated",
        r"synthetic\s+(?:content|media|image)",
        r"\bdisclos\w*",
        r"label(?:l)?(?:ed|ing)?\s+as\s+(?:ai|synthetic)",
    ],
    "prohibited_uses": [
        r"\bdeepfake",
        r"\bimpersonat",
        r"non[- ]consensual",
        r"\b(?:dis|mis)information\b",
        r"prohibited\s+use",
        r"out[- ]of[- ]scope\s+use",
        r"\bmisuse\b",
    ],
    "downstream_obligation": [
        r"downstream\s+(?:user|developer|deployer|application)",
        r"(?:users?|deployers?|developers?)\s+(?:must|should|are\s+(?:required|expected))",
        r"you\s+(?:must|are\s+responsible)",
    ],
    "regulatory_reference": [
        r"\bai\s+act\b",
        r"\bgdpr\b",
        r"\bdsa\b",
        r"\blegal\s+(?:obligation|requirement)",
        r"\bregulatory\s+(?:requirement|obligation|compliance)",
    ],
}

# Context cues that disqualify a candidate match.
EXCLUSIONS = {
    "watermark": [
        r"negative[_ ]prompt",
        r"watermark\s+probability",          # LAION training-data filtering
        r"estimated\s+watermark",
        r"filtered\s+to\s+images",
        r"pip\s+install[^\n]{0,80}watermark",  # dependency line, not a claim
        r"```[^`]{0,60}watermark[^`]{0,60}```",
        r"low\s*res|lowres|worst\s+quality|low\s+quality|jpeg\s+artifacts",
        r"\bsignature\b.{0,40}\busername\b|\busername\b.{0,40}\bsignature\b",
        r"\bblurry\b|\bdeformed\b|\bmutated\b",
        r"remove\s+watermark|watermark\s+removal",
        r"generat\w+\s+(?:in-image\s+)?text.{0,60}watermark",
        r"logos?,\s*captions?,\s*watermarks?",
    ],
    "provenance": [
        r"(?:data|dataset|corpus|training)\s+provenance",
        r"provenance\s+(?:tracking|filtering)\s+(?:are|is)\s+applied\s+across\s+the\s+corpus",
        r"deduplication\s+and\s+provenance",
        r"data\s+sources?\s+are\s+reviewed",
        r"provenance,?\s+and\s+alignment\s+with",
        r"licensing\s+compatibility",
        r"admission\s+into\s+training\s+corpora",
    ],
    "synthetic_disclosure": [
        r"synthetic\s+(?:images?|data|samples?)\s+(?:generated\s+)?(?:using|from|with|by)",
        r"ai[- ]generated\s+look",
        r"reduces?\s+the\s+.{0,20}ai[- ]generated",
        r"\|\s*\d+[MK]?\s*\|",          # training-data size tables
        r"training\s+(?:data|set|corpus)",
    ],
    # Red-teaming and evaluation descriptions name harm categories without
    # stating a prohibition. Validation found three such false positives.
    "prohibited_uses": [
        r"testing\s+covered",
        r"red[- ]team",
        r"resilience\s+to\s+attempts",
        r"based\s+on\s+these\s+evaluations",
        r"testing\s+was\s+conducted",
    ],
    # Gate and authentication instructions are access mechanics, not
    # obligations on downstream conduct. Validation found two.
    "downstream_obligation": [
        r"accept\s+the\s+gate",
        r"huggingface[- ]cli\s+login",
        r"HF_TOKEN",
        r"authenticate\s+before",
        r"gated\s+(?:model|repo)",
    ],
    "regulatory_reference": [],
}

# Context cues that confirm a candidate, overriding an exclusion.
CONFIRMATIONS = {
    "watermark": [
        r"implements?\s+[^.]{0,40}watermark",
        r"pixel[- ]layer\s+watermark",
        r"applies?\s+[^.]{0,60}(?:watermark|c2pa)",
        r"output.{0,60}watermark",
        r"watermark.{0,60}output",
        r"\bsynthid\b",
        r"invisible\s+watermark.{0,80}(?:embed|appl|add|attach)",
        r"(?:embed|appl|add|attach)\w*.{0,60}invisible\s+watermark",
        r"watermark\w*\s+(?:is|are)\s+(?:embedded|applied|added)",
        r"all\s+(?:images?|outputs?).{0,40}watermark",
    ],
    "provenance": [r"\bc2pa\b", r"content\s+credential", r"content\s+authenticity"],
    "synthetic_disclosure": [
        r"(?:identify|label|interpret|detect)[^.]{0,60}ai[- ]generated",
        r"indicate[^.]{0,60}(?:ai|synthetic|generated)",
        r"disclos\w+.{0,80}(?:ai|synthetic|generated)",
        r"(?:ai|synthetic|generated).{0,80}disclos\w+",
        r"label(?:l)?(?:ed|ing)?\s+as\s+(?:ai|synthetic)",
        r"must\s+(?:be\s+)?(?:marked|labelled|labeled|indicated)",
    ],
    "prohibited_uses": [],
    "downstream_obligation": [],
    "regulatory_reference": [],
}

# Categories where a bare keyword match is too ambiguous to count. A candidate
# is accepted only if an output-directed confirmation cue appears in context.
REQUIRE_CONFIRMATION = {"watermark", "provenance", "synthetic_disclosure"}

C_CAND = {k: [re.compile(p, re.I) for p in v] for k, v in CANDIDATES.items()}
C_EXCL = {k: [re.compile(p, re.I) for p in v] for k, v in EXCLUSIONS.items()}
C_CONF = {k: [re.compile(p, re.I) for p in v] for k, v in CONFIRMATIONS.items()}


def _classify(body: str, cat: str) -> tuple[bool, bool, str | None]:
    """Returns (raw_hit, refined_hit, evidence span)."""
    raw_hit = False
    evidence = None
    for pat in C_CAND[cat]:
        for m in pat.finditer(body):
            lo = max(0, m.start() - WINDOW)
            hi = min(len(body), m.end() + WINDOW)
            ctx = body[lo:hi]
            if not raw_hit:
                raw_hit = True
                evidence = ctx.replace("\n", " ").strip()
            confirmed = any(c.search(ctx) for c in C_CONF[cat])
            if confirmed:
                return True, True, ctx.replace("\n", " ").strip()
            if cat in REQUIRE_CONFIRMATION:
                continue
            if any(e.search(ctx) for e in C_EXCL[cat]):
                continue
            return True, True, ctx.replace("\n", " ").strip()
    return raw_hit, False, evidence


CATEGORIES = list(CANDIDATES)


def code_card_v2(text) -> dict:
    body = text if isinstance(text, str) else ""
    out: dict = {"card_chars": len(body), "has_card": len(body.strip()) > 200}
    for cat in CATEGORIES:
        raw, refined, ev = _classify(body, cat)
        out[f"{cat}_raw"] = raw
        out[cat] = refined
        out[f"{cat}_evidence"] = ev
    # Dependency-only: the card names a watermarking package in an install or
    # code block but makes no claim about marking outputs. Counted separately,
    # because an install line is not a disclosure to downstream developers.
    import re as _re
    out["watermark_dependency_only"] = bool(
        (not out["watermark"]) and out["watermark_raw"] and
        _re.search(r"(?:pip\s+install|import)[^\n]{0,80}watermark", body, _re.I))
    out["any_provenance_signal"] = bool(
        out["watermark"] or out["provenance"] or out["synthetic_disclosure"])
    out["any_provenance_signal_raw"] = bool(
        out["watermark_raw"] or out["provenance_raw"] or out["synthetic_disclosure_raw"])
    return out


In [ ]:
def download_tier(n):
    try: n = int(n)
    except (TypeError, ValueError): return "unknown"
    if n >= 1_000_000: return "1M+"
    if n >= 100_000:   return "100k-1M"
    if n >= 10_000:    return "10k-100k"
    if n >= 1_000:     return "1k-10k"
    return "<1k"

def license_group(lic):
    if not isinstance(lic, str) or not lic.strip(): return "unspecified"
    l = lic.lower()
    if any(k in l for k in ("creativeml","openrail","rail")): return "RAIL family"
    if "-nc" in l or "noncommercial" in l:                    return "non-commercial"
    if any(k in l for k in ("apache","mit","bsd","cc-by","cc0")): return "permissive"
    return "other"

## 4. Code the cards

In [ ]:
coded = pd.DataFrame([code_card_v2(t) for t in raw["card_text"]])
df = pd.concat([raw.drop(columns=["card_text"]), coded], axis=1)
df["card_text"]     = raw["card_text"]
df["license"]       = df["tags"].fillna("").str.extract(r"license:([^,]+)")[0]
df["license_group"] = df["license"].apply(license_group)
df["download_tier"] = df["downloads"].apply(download_tier)
df["base_model"]    = df["tags"].fillna("").str.extract(
    r"base_model:(?:finetune:|adapter:|quantized:|merge:)?([^,]+)")[0]
df["is_derivative"] = df["base_model"].notna()

df.drop(columns=["card_text"]).to_csv(OUT / "coded.csv", index=False)
print(f"N = {len(df)} | substantive card {df.has_card.mean():.1%}"
      f" | derivatives {df.is_derivative.mean():.1%}")

## 5. Table 1: naive versus validated coding

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    import math
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return (max(0,c-h), min(1,c+h))

rows=[]
for c in CATEGORIES:
    k, n = int(df[c].sum()), len(df)
    lo, hi = wilson(k, n)
    rows.append({"category": c, "naive_rate": df[f"{c}_raw"].mean(),
                 "final_rate": k/n, "ci_low": lo, "ci_high": hi,
                 "fp_removed": 1 - (k/max(df[f"{c}_raw"].sum(),1))})
t1 = pd.DataFrame(rows).sort_values("final_rate", ascending=False)
print(t1.to_string(index=False, float_format=lambda x: f"{x:0.3f}"))
t1.to_csv(OUT/"table1_coding.csv", index=False)

k = int(df.any_provenance_signal.sum()); lo, hi = wilson(k, len(df))
print(f"\nexplicit output-marking disclosure: {k}/{len(df)} = {k/len(df):.1%}"
      f"  [95% CI {lo:.1%}-{hi:.1%}]")
print(f"naive coding would report: {df.any_provenance_signal_raw.mean():.1%}")
print(f"regulatory references: {int(df.regulatory_reference.sum())}")

## 6. Table 2: who discloses, and how concentrated is it

In [ ]:
sig = df[df.any_provenance_signal].sort_values("downloads", ascending=False)
print(sig[["model_id","downloads","watermark","provenance","synthetic_disclosure"]]
      .head(30).to_string(index=False))
print(f"\nn = {len(sig)} | download share {sig.downloads.sum()/df.downloads.sum():.1%}")
print(f"organizations: {sig.org.nunique()} -> {sorted(sig.org.unique())[:15]}")
sig.drop(columns=["card_text"]).to_csv(OUT/"table2_disclosers.csv", index=False)

## 7. Table 3: does disclosure survive derivation?

In [ ]:
t3 = df.groupby("is_derivative").agg(
    n=("model_id","size"),
    explicit_disclosure=("any_provenance_signal","mean"),
    prohibited_uses=("prohibited_uses","mean"),
    downstream_obligation=("downstream_obligation","mean"))
t3.index = ["upstream (no declared base)", "derivative"]
print(t3.to_string(float_format=lambda x: f"{x:0.3f}"))
t3.to_csv(OUT/"table3_lineage.csv")

disclosing_orgs = [o.lower() for o in sig.org.unique()]
pat = "|".join(disclosing_orgs + ["flux","stable-diffusion","stabilityai","compvis"])
child = df.is_derivative & df["base_model"].fillna("").str.lower().str.contains(pat)
print(f"\nderivatives of a disclosing or major base: {int(child.sum())}")
if child.sum():
    print(f"  disclose themselves: {int(df[child].any_provenance_signal.sum())}"
          f" ({df[child].any_provenance_signal.mean():.1%})")
    silent = df[child & ~df.any_provenance_signal]
    print(f"  downloads of the silent ones: {silent.downloads.sum():,}")

## 8. Second-coder validation

The schema was validated once at N=100 (73 rows, precision 0.973) by the
schema's author. Regenerate the sheet at this sample size and code it yourself,
then report agreement between the two coders rather than one coder's precision.

Every positive in the three rare categories is included, plus every rejected
candidate, so the exclusion logic is checked too.

In [ ]:
RARE = ["watermark","provenance","synthetic_disclosure"]
COMMON = ["prohibited_uses","downstream_obligation","regulatory_reference"]
recs=[]
for c in RARE:
    for _,r in df[df[c]].iterrows():
        recs.append({"priority":"A-rare-positive","model_id":r.model_id,
                     "category":c,"evidence":r[f"{c}_evidence"],"correct":""})
    for _,r in df[df[f"{c}_raw"] & ~df[c]].iterrows():
        recs.append({"priority":"B-rare-rejected","model_id":r.model_id,
                     "category":c+" (REJECTED)","evidence":r[f"{c}_evidence"],"correct":""})
for c in COMMON:
    g = df[df[c]]
    for _,r in g.sample(min(len(g),15), random_state=3).iterrows():
        recs.append({"priority":"C-common-positive","model_id":r.model_id,
                     "category":c,"evidence":r[f"{c}_evidence"],"correct":""})
neg = df[~df[CATEGORIES].any(axis=1)]
for _,r in neg.sample(min(len(neg),25), random_state=3).iterrows():
    txt = r.card_text if isinstance(r.card_text,str) else ""
    recs.append({"priority":"D-negative","model_id":r.model_id,
                 "category":"(none flagged)","evidence":txt[:600],"correct":""})
vs = pd.DataFrame(recs)
vs.to_csv(OUT/"validation_sheet.csv", index=False)
print(vs.priority.value_counts().to_string()); print("total rows:", len(vs))

In [ ]:
v = pd.read_csv(OUT/"validation_sheet.csv")
v = v[v["correct"].astype(str).isin(["0","1"])]
if len(v):
    v["correct"] = v["correct"].astype(int)
    p = v.groupby("priority")["correct"].agg(["mean","count"]); p.columns=["precision","n"]
    print(p.to_string(float_format=lambda x: f"{x:0.3f}"))
    print(f"\noverall {v.correct.mean():.3f} across {len(v)} rows")
    p.to_csv(OUT/"coder_precision.csv")
else:
    print("Nothing marked yet.")

## 9. Figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))
o = t1.sort_values("final_rate"); y = range(len(o))
axes[0].barh(y, o["naive_rate"], color="#c9d3e3", label="naive keyword coding")
axes[0].barh(y, o["final_rate"], color="#2f4b7c", label="validated coding")
axes[0].set_yticks(list(y)); axes[0].set_yticklabels(o["category"].str.replace("_"," "))
axes[0].set_xlim(0,1); axes[0].set_xlabel(f"Share of models (N={len(df)})")
axes[0].set_title("Disclosure coverage, and what naive coding overstates")
axes[0].legend(frameon=False, fontsize=8)

axes[1].bar(["upstream","derivative"], t3["explicit_disclosure"], color=["#2f4b7c","#bc5090"])
axes[1].set_ylabel("Share with explicit output-marking disclosure")
axes[1].set_title("Disclosure does not survive derivation")
for i, v_ in enumerate(t3["explicit_disclosure"]):
    axes[1].text(i, v_+0.003, f"{v_:.1%}", ha="center", fontsize=9)
fig.tight_layout(); fig.savefig(OUT/"figure1.png", dpi=200); plt.show()

## 10. Export

In [ ]:
import shutil
shutil.make_archive("mcd_out","zip",OUT)
try:
    from google.colab import files; files.download("mcd_out.zip")
except Exception:
    print("grab mcd_out.zip from the file browser")

## 11. Limitations to state in the paper

1. **Sample frame.** Most-downloaded text-to-image models on one hub, fetched on
   a recorded date. Popularity-weighted, single modality, single channel.
2. **Schema development.** The schema was revised twice after inspecting results
   on the N=100 sample. Say so, and report the version used for the final run.
3. **Validation.** Precision 0.973 on 73 rows, coded by the schema's author.
   Second-coder agreement is stronger and section 8 regenerates the sheet for it.
4. **Absence is not silence.** A card that says nothing about marking does not
   prove the model lacks it. The claim concerns what is *communicated to
   downstream developers*, which is what a disclosure obligation depends on.
5. **Cards change.** `raw_cards.jsonl` is the archive that makes the run
   reproducible.